# Bring Your Own Model

Evaluate whether your RNA/DNA foundation model encodes genuine secondary structure.

Three rungs, each harder than the last:  
**Rung 1** — Can the model tell stem positions from loop positions, after controlling for base composition?  
**Rung 2** — Does that signal survive dinucleotide-level controls?  
**Rung 3** — Does the model know *which specific position pairs with which*?

In [ ]:
# Install (run once)
!pip install -q rna-structure-audit transformers torch

## The adapter interface

You write a thin wrapper around your model — three methods, that's it.

```python
class ModelAdapter(ABC):
    name: str       # display name
    d_model: int    # hidden dimension
    n_layers: int   # number of layers

    def load(self):                           # load weights
    def tokenize(self, sequence: str):         # str -> input_ids tensor
    def get_all_layer_embeddings(self, tokens): # tokens -> list of (seq_len, d_model) tensors
```

The key constraint: `get_all_layer_embeddings` must return **nucleotide-resolution** embeddings with special tokens (CLS, EOS) stripped.

## Write your adapter

Here's a generic pattern that works for most HuggingFace models with `output_hidden_states=True`. Replace the model name and adjust `n_layers`/`d_model` to match your model.

In [ ]:
import torch
from rna_structure_audit.adapter import ModelAdapter


class MyModelAdapter(ModelAdapter):
    # ---- Change these to match your model ----
    name = "my-rna-model"
    d_model = 768
    n_layers = 12
    HF_MODEL_ID = "your-org/your-model"  # HuggingFace model ID
    # -------------------------------------------

    def __init__(self):
        self.model = None
        self.tokenizer = None

    def load(self):
        from transformers import AutoModel, AutoTokenizer

        self.tokenizer = AutoTokenizer.from_pretrained(
            self.HF_MODEL_ID, trust_remote_code=True
        )
        self.model = AutoModel.from_pretrained(
            self.HF_MODEL_ID, trust_remote_code=True
        )
        self.model.eval()

    def tokenize(self, sequence: str) -> torch.Tensor:
        enc = self.tokenizer(sequence, return_tensors="pt")
        return enc["input_ids"]

    @torch.no_grad()
    def get_all_layer_embeddings(
        self, tokens: torch.Tensor
    ) -> list[torch.Tensor]:
        out = self.model(tokens, output_hidden_states=True)
        # Strip CLS and EOS tokens (positions 0 and -1)
        # Adjust the slice if your model uses different special tokens
        return [hs[0, 1:-1, :] for hs in out.hidden_states]

## Quick sanity check: run Rung 3 only on one family

Before running the full evaluation (~10 min on a T4), test that your adapter works.

In [ ]:
adapter = MyModelAdapter()
adapter.load()

# Quick test: tokenize a short RNA and get embeddings
test_seq = "GGCUAGCCUAAGCC"
tokens = adapter.tokenize(test_seq)
embeddings = adapter.get_all_layer_embeddings(tokens.to(next(adapter.model.parameters()).device))

print(f"Layers returned: {len(embeddings)}")
print(f"Shape per layer: {embeddings[0].shape}")

assert len(embeddings) == adapter.n_layers + 1, (
    f"Expected {adapter.n_layers + 1} layers (embedding + {adapter.n_layers} transformer), "
    f"got {len(embeddings)}. Adjust n_layers."
)
assert embeddings[0].shape[0] == len(test_seq), (
    f"Expected seq_len={len(test_seq)}, got {embeddings[0].shape[0]}. "
    f"Make sure you strip special tokens in get_all_layer_embeddings."
)
print("Adapter looks good.")

## Run the full evaluation

In [ ]:
from rna_structure_audit.evaluate import evaluate

device = "cuda" if torch.cuda.is_available() else "cpu"
results = evaluate(adapter, device=device, output_path="my_model_results.json")

report = results["report"]
print(f"\nModel: {report['model']}")
print(f"Grade: {report.get('grade', 'N/A')} — {report.get('grade_description', '')}")

## Interpret the results

| Grade | Meaning |
|-------|---------|
| **A** | Partner-specific: the model knows which position pairs with which |
| **B** | Structure-aware beyond composition: survives dinucleotide controls |
| **C** | Composition-sensitive: stem/loop signal is absorbed by nucleotide frequency controls |
| **D** | No detectable structure signal |

The key metric at Rung 3 is **perturbation specificity (PS)**: when you mutate position *i*, does the model preferentially perturb position *i*'s pairing partner over adjacent stem positions? PS > 0.05 with precision > 50% earns an A.

In [ ]:
if "rung1" in report:
    r1 = report["rung1"]
    print(f"Rung 1: stem/loop ratio = {r1['mean_ratio']:.2f}, "
          f"{r1['families_exceeding_null']}/{r1['families_scored']} families > null")

if "rung2" in report:
    r2 = report["rung2"]
    print(f"Rung 2: {r2['surviving_dinuc']}/{r2['rung1_passing']} survive dinucleotide null")

if "rung3" in report:
    r3 = report["rung3"]
    print(f"Rung 3: PS = {r3['mean_ps']:.4f}, "
          f"precision = {r3['mean_h3_precision']:.1%}, "
          f"{r3['exceeding_null']}/{r3['gate_passing']} families > null")

## How does your model compare?

Results from our evaluation of 10 models (Tower, 2026):

| Model | Params | Pretrained on | Grade | PS |
|-------|--------|---------------|-------|----|
| RiNALMo | 650M | RNA | **A** | 0.215 |
| ERNIE-RNA | 86M | RNA | **A** | 0.117 |
| Caduceus | 14M | DNA | D | 0.004 |
| Evo | 7B | DNA | D | 0.001 |
| SpliceBERT | 19M | RNA | D | 0.0004 |
| UTR-LM | 2M | RNA | D | 0.00001 |
| HyenaDNA | 5.4M | DNA | D | ~0 |
| RNA-FM | 99M | RNA | D | ~0 |
| NT v2 | 56M | DNA | D | ~0 |
| DNABERT-2 | 117M | DNA | D | -0.016 |

Only two models achieve Grade A. Both are RNA-pretrained; DNA-pretrained models produce near-zero partner specificity regardless of scale.

## Tips

- **BPE tokenizers**: If your model uses BPE or k-mer tokenization, embeddings won't be at nucleotide resolution. The benchmark handles this but many families will be skipped.
- **Large models**: For models > 1B params, you may need an A100. Free Colab T4s (16GB) can handle up to ~650M.
- **Running a subset**: Use `evaluate(adapter, rungs=(3,))` to skip Rungs 1-2 and go straight to partner specificity.
- **Saving results**: Pass `output_path="results.json"` to save the full JSON for later analysis.

---

## Bring your own data

The bundled benchmark uses 52 Rfam families. You can also run the evaluation on your own RNA sequences — all you need is a sequence and its dot-bracket secondary structure.

Each family is a dict with three required fields:

```python
{"name": "my_rna", "sequence": "GGCUAGCC...", "dot_bracket": "(((...)))..."}
```

Sequences must be at least 20 nt, with at least 5 paired and 5 unpaired positions.

### Option A: Define families inline

In [ ]:
my_families = [
    {
        "name": "my_tRNA",
        "sequence": "GCCGAUAUAGUGUAACGGCUAUCACAUCACGCUUUCACCGUGGAGACCGGGGUUCGACUCCCCGUAUCGGCA",
        "dot_bracket": "(((((((..((((.......)))).(((((.......)))))..(((((.......))))).))))))....",
    },
    {
        "name": "my_hairpin",
        "sequence": "GGGGAAAACCCCUUUUAAAAGGGGUUUU",
        "dot_bracket": "((((....))))....((((....))))",
    },
]

### Option B: Load from JSON files

You can also save families as individual JSON files and load them from a directory.

In [ ]:
import json
from pathlib import Path

# Save example families to disk
data_dir = Path("my_rna_data")
data_dir.mkdir(exist_ok=True)

for fam in my_families:
    with open(data_dir / f"{fam['name']}.json", "w") as f:
        json.dump(fam, f, indent=2)

# Load them back
custom_families = []
for p in sorted(data_dir.glob("*.json")):
    with open(p) as f:
        custom_families.append(json.load(f))

print(f"Loaded {len(custom_families)} families: {[f['name'] for f in custom_families]}")

### Run the evaluation on custom data

Call the rung functions directly with your families list. Here we run Rung 3 (partner specificity) — the most informative single rung.

In [ ]:
from rna_structure_audit.rungs import run_rung1, run_rung2, run_rung3

# Use a pre-built adapter (or your own from above)
from rna_structure_audit.adapters import ERNIERNAAdapter

adapter = ERNIERNAAdapter()
adapter.load()

device = "cuda" if torch.cuda.is_available() else "cpu"

# Run all three rungs on your custom families
r1 = run_rung1(adapter, my_families, device=device)
r2 = run_rung2(adapter, my_families, r1, device=device)
r3 = run_rung3(adapter, my_families, device=device)

print(f"Rung 1: {r1['n_exceeding_null']}/{r1['n_families_scored']} families exceed null")
print(f"Rung 2: {r2['n_surviving_dinuc']}/{r2['n_rung1_passing']} survive dinucleotide null")
print(f"Rung 3: mean PS = {r3['mean_ps']:.4f}")

# Per-family breakdown
for name, res in r3["per_family"].items():
    if "skipped" not in res:
        print(f"  {name}: PS = {res['ps']:.4f}")

### Where to get dot-bracket structures

If you have a sequence but no known structure:

- **RNAfold** (ViennaRNA): `RNAfold < seq.fasta` gives the MFE structure
- **Rfam**: search your sequence at [rfam.org](https://rfam.org) — curated families include consensus structures
- **PDB/NDB**: experimentally determined structures can be converted to dot-bracket with `x3dna-dssr`
- **bpRNA**: pre-parsed dot-bracket annotations for thousands of RNAs from multiple databases

For the benchmark to be meaningful, the structure should be experimentally validated or from a well-established consensus. MFE predictions work for testing your pipeline but may not reflect the true structure.